# Нейросети I: многослойный перцептрон (MLP)

## Содержание

0. Подготовка среды и подключение Weights & Biases
1. Данные: вернёмся к оттоку клиентов
2. Нейрон и слой: математический аппарат
3. Первая нейросеть на PyTorch
4. Обучение: эпохи и функция потерь
5. Оценка и сравнение с классическими моделями
6. Практикум

Это первое занятие про нейросети. Мы не будем сразу строить что-то сложное: разберём базовый «кирпичик» — **нейрон и слой**, соберём маленькую полносвязную сеть (**MLP**), обучим её и сравним с логистической регрессией из занятия 3. Идея в том, чтобы увидеть, что нейросеть — это знакомые вам вещи (линейная комбинация + нелинейность + градиентный спуск), просто собранные вместе.

Демонстрационный пример — тот же прогноз оттока клиентов, что и в занятии 3. Практикум (раздел 6) вы выполняете на **датасете своего проекта**. Выводы после каждого блока записывайте **в ячейке markdown прямо в блокноте**.

## 0. Подготовка среды и подключение Weights & Biases

Для этого занятия нужен **PyTorch**. В Google Colab он уже установлен. Локально: `pip install torch`. Это занятие можно целиком выполнять на CPU — сеть маленькая.

**Примечание.** Если с W&B проблемы, замените `wandb.init(...)` на `wandb.init(mode="disabled")`.

In [ ]:
%pip install -q wandb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

import wandb

torch.manual_seed(42)
RANDOM_STATE = 42
plt.rcParams['figure.figsize'] = (8, 5)
wandb.login()

## 1. Данные: вернёмся к оттоку клиентов

Берём тот же датасет, что и в занятии 3 (Telco Customer Churn). Подготовка такая же, как раньше: пропуски в числовых признаках заполняем медианой, числовые стандартизуем, категориальные кодируем one-hot. Весь препроцессинг — внутри `Pipeline`, чтобы не было утечки.

**Важно для нейросетей:** они работают с числами и чувствительны к масштабу, поэтому предобработка не опциональна, а обязательна.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

d = df.drop(columns=['customerID']).copy()
d['TotalCharges'] = pd.to_numeric(d['TotalCharges'], errors='coerce')
d['Churn'] = (d['Churn'] == 'Yes').astype(int)

num = [c for c in d.select_dtypes(include=np.number).columns if c != 'Churn']
cat = [c for c in d.columns if c not in num + ['Churn']]
X, y = d[num + cat], d['Churn'].to_numpy()

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE)

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat),
])

def dense(M):
    return (M.toarray() if hasattr(M, 'toarray') else np.asarray(M)).astype('float32')

Xtr = dense(prep.fit_transform(X_train))
Xval = dense(prep.transform(X_val))
Xte = dense(prep.transform(X_test))
print("Обучающая:", Xtr.shape, "| признаки:", Xtr.shape[1])

In [ ]:
def to_dataset(X, y):
    return TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.float32).unsqueeze(1))

train_loader = DataLoader(to_dataset(Xtr, y_train), batch_size=256, shuffle=True)
Xval_t = torch.tensor(Xval)
Xte_t = torch.tensor(Xte)
print("Батчей в эпохе:", len(train_loader))

## 2. Нейрон и слой: математический аппарат

**Нейрон** — это линейная комбинация входов, пропущенная через нелинейную функцию активации:

$$z = \sum_i w_i x_i + b, \qquad a = f(z)$$

Функция активации делает модель нелинейной. Чаще всего берут **ReLU**: $f(z) = \max(0, z)$. На выходе для бинарной классификации — **сигмоида** $\sigma(z) = \frac{1}{1+e^{-z}}$, и результат читается как вероятность (как в логистической регрессии из занятия 3).

**Слой** — это сразу много нейронов. Удобно записать матрично: $a = f(Wx + b)$, где $W$ — матрица весов, $b$ — вектор смещений. **MLP (многослойный перцептрон)** — это несколько таких слоёв подряд: выход одного становится входом следующего.

**Обучение** ничем не отличается от того, что было раньше:

- функция потерь для бинарной классификации — **бинарная кросс-энтропия** (та же, что в занятии 3);
- веса подбираются **градиентным спуском**, но не по всем данным сразу, а по **мини-батчам**;
- производные по весам считает **алгоритм обратного распространения (backprop)** — это правило цепочки, которое вы проходили в курсе 1. В PyTorch его считает `loss.backward()`, вручную писать не нужно.

**Задача 1.**
Покажите, что если убрать функции активации, то два линейных слоя подряд эквивалентны одному линейному слою. Почему без нелинейности нейросеть не может моделировать ничего, кроме линейной зависимости?

**Ваш вывод:**

Без активации композиция двух слоёв имеет вид `W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂)`. Это один линейный слой с другими весами и смещением. Поэтому сколько бы линейных слоёв мы ни добавили, без нелинейной активации сеть останется линейной моделью и не сможет описывать нелинейные зависимости.

## 3. Первая нейросеть на PyTorch

Соберём простую сеть: один входной слой, один скрытый слой на 16 нейронов с ReLU и один выходной нейрон. Для бинарной классификации удобно использовать `BCEWithLogitsLoss` — она объединяет сигмоиду и кросс-энтропию и численно устойчива.

In [ ]:
n_features = Xtr.shape[1]

model = nn.Sequential(
    nn.Linear(n_features, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(model)

## 4. Обучение: эпохи и функция потерь

**Эпоха** — один полный проход по обучающим данным. Внутри эпохи идут мини-батчи: для каждого считается loss, берётся градиент и делается шаг оптимизатора. Будем сохранять средний loss по эпохам и следить за кривой обучения.

In [ ]:
epochs = 30
history = []

for epoch in range(epochs):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    history.append(total / len(Xtr))
    if (epoch + 1) % 10 == 0:
        print(f"эпоха {epoch + 1}: train loss = {history[-1]:.3f}")

plt.plot(range(1, epochs + 1), history)
plt.xlabel('эпоха')
plt.ylabel('train loss')
plt.title('Кривая обучения')
plt.show()

**Примечание.** Loss убывает — сеть учится. Если кривая выходит на плато и не падает, попробуйте увеличить число эпох; если loss на train падает, а качество на validation нет — сеть переобучается (в следующем занятии разберём, как с этим бороться: dropout, ранняя остановка и т.д.).

## 5. Оценка и сравнение с классическими моделями

Получаем вероятности на тесте (не забудьте сигмоиду, потому что модель выдаёт логиты) и считаем метрики из занятия 3: **Average Precision** и **ROC-AUC**. Сравним с логистической регрессией на тех же данных.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(Xte_t).numpy().ravel()
proba = 1 / (1 + np.exp(-logits))

mlp_ap = average_precision_score(y_test, proba)
mlp_roc = roc_auc_score(y_test, proba)

lr = LogisticRegression(C=0.3, max_iter=1500, solver='liblinear').fit(Xtr, y_train)
lr_proba = lr.predict_proba(Xte)[:, 1]

print(f"MLP: AP = {mlp_ap:.3f}, ROC-AUC = {mlp_roc:.3f}")
print(f"LogReg: AP = {average_precision_score(y_test, lr_proba):.3f}, ROC-AUC = {roc_auc_score(y_test, lr_proba):.3f}")

run = wandb.init(project="apml-course", name="mlp-churn", config={"hidden": 16, "epochs": epochs, "lr": 0.01})
wandb.log({"mlp_ap": mlp_ap, "mlp_roc_auc": mlp_roc})
run.finish()

**Задача 2.**
Поменяйте число нейронов в скрытом слое (например, 4, 16, 64) и число эпох. Как меняется качество и кривая обучения? Сравните нейросеть с логистической регрессией: заметен ли выигрыш и почему?

In [ ]:
scores = []
for h in [4, 16, 64]:
    net = nn.Sequential(nn.Linear(Xtr.shape[1], h), nn.ReLU(), nn.Linear(h, 1))
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for _ in range(30):
        net.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        p = torch.sigmoid(net(Xval_t)).numpy().ravel()
    ap = average_precision_score(y_val, p)
    auc = roc_auc_score(y_val, p)
    scores.append([h, ap, auc])
    run = wandb.init(project="apml-course", name=f"mlp-churn-{h}", config={"hidden": h, "epochs": 30, "lr": 0.01})
    wandb.log({"val_ap": ap, "val_roc_auc": auc})
    run.finish()

lr = LogisticRegression(C=0.3, max_iter=1500, solver='liblinear').fit(Xtr, y_train)
p = lr.predict_proba(Xval)[:, 1]
scores.append(["LogReg", average_precision_score(y_val, p), roc_auc_score(y_val, p)])
pd.DataFrame(scores, columns=["hidden", "val_AP", "val_ROC_AUC"])

**Вывод:**

Сравнение размеров сети делаю по AP и ROC-AUC на validation. Увеличение скрытого слоя повышает гибкость модели, но не гарантирует улучшения: после некоторого размера качество может выйти на плато или начать снижаться. Число эпох выбираю по validation, а не только по уменьшению train loss. Логистическая регрессия остаётся простой базовой моделью для сравнения.

## 6. Практикум

Задания выполняются на **датасете своего проекта**. Нейросеть — ещё одна модель для вашей задачи классификации (или регрессии). После каждого задания заполните ячейку **«Вывод»**.

### Задание 6.1. Подготовка данных

Возьмите числовые признаки (категориальные закодируйте one-hot), обработайте пропуски и стандартизуйте. Разделите данные на train/val/test со стратификацией и переведите в тензоры и батчи. Сформулируйте целевую переменную.

In [ ]:
from pathlib import Path

path = "housing.csv" if Path("housing.csv").exists() else "housing(1).csv"
df = pd.read_csv(path)
target = "median_house_value"
X = df.drop(columns=target)
y = df[target].to_numpy(dtype="float32") / 100000
bins = pd.qcut(df[target], q=10, labels=False, duplicates="drop")

X_train, X_tmp, y_train, y_tmp, b_train, b_tmp = train_test_split(
    X, y, bins, test_size=0.4, stratify=bins, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=b_tmp, random_state=RANDOM_STATE
)

num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])

def dense(M):
    return (M.toarray() if hasattr(M, "toarray") else np.asarray(M)).astype("float32")

Xtr = dense(prep.fit_transform(X_train))
Xval = dense(prep.transform(X_val))
Xte = dense(prep.transform(X_test))
train_loader = DataLoader(
    TensorDataset(torch.tensor(Xtr), torch.tensor(y_train).unsqueeze(1)),
    batch_size=256, shuffle=True
)
Xval_t = torch.tensor(Xval)
Xte_t = torch.tensor(Xte)
yval_t = torch.tensor(y_val).unsqueeze(1)
print(Xtr.shape, Xval.shape, Xte.shape)
print("Train / val / test:", len(y_train), len(y_val), len(y_test))
print("Числовых признаков:", len(num), "| категориальных:", len(cat))

**Вывод:**

Цель — `median_house_value`, поэтому это задача регрессии. Получилось 12 384 объекта для обучения, по 4 128 для validation и test. Числовые признаки заполнены медианой и стандартизованы, `ocean_proximity` закодирован one-hot. Для стратификации цены разделил на 10 квантильных групп; `total_bedrooms` содержит пропуски, которые заполняются внутри pipeline.

### Задание 6.2. Обучение MLP

Соберите простой MLP (один-два скрытых слоя), задайте функцию потерь и оптимизатор. Обучите его и постройте кривую обучения. Подберите число эпох по качеству на validation.

In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(Xtr.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
run = wandb.init(
    project="apml-course",
    name="mlp-housing",
    config={"hidden": [128, 64, 32], "epochs": 180, "batch_size": 256, "lr": 0.001}
)

best_val = np.inf
best_epoch = 0
best_state = None
wait = 0
history = []

for epoch in range(180):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    train_loss = total / len(Xtr)
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(Xval_t), yval_t).item()
    history.append((train_loss, val_loss))
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss, "val_rmse": np.sqrt(val_loss) * 100000})
    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_epoch = epoch + 1
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
    if wait == 15:
        break

model.load_state_dict(best_state)
print("Best epoch:", best_epoch, "| val RMSE:", round(np.sqrt(best_val) * 100000))
plt.plot([x[0] for x in history], label="train")
plt.plot([x[1] for x in history], label="validation")
plt.xlabel("эпоха")
plt.ylabel("MSE")
plt.legend()
plt.show()

**Вывод:**

Использовал MLP с тремя скрытыми слоями на 128, 64 и 32 нейрона. Лучший результат на validation получился на 98-й эпохе; после 15 эпох без улучшения обучение остановилось на 113-й. Это помогает не продолжать обучение, когда качество на новых данных уже не растёт.

### Задание 6.3. Оценка и сравнение

Посчитайте метрики на test (AP, ROC-AUC, а для регрессии — RMSE/R²) и сравните нейросеть с логистической регрессией или бустингом из предыдущих занятий. Залогируйте запуск в W&B.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

with torch.no_grad():
    mlp_pred = model(Xte_t).numpy().ravel() * 100000
y_true = y_test * 100000

boost = HistGradientBoostingRegressor(
    max_iter=250, learning_rate=0.08, max_leaf_nodes=31,
    l2_regularization=1, random_state=RANDOM_STATE
)
boost.fit(Xtr, y_train * 100000)
boost_pred = boost.predict(Xte)

mlp_rmse = mean_squared_error(y_true, mlp_pred) ** 0.5
mlp_r2 = r2_score(y_true, mlp_pred)
boost_rmse = mean_squared_error(y_true, boost_pred) ** 0.5
boost_r2 = r2_score(y_true, boost_pred)

results = pd.DataFrame([
    ["MLP", mlp_rmse, mlp_r2],
    ["HistGradientBoosting", boost_rmse, boost_r2],
], columns=["Модель", "RMSE", "R2"])
print(results.round(3))
wandb.log({
    "best_epoch": best_epoch,
    "mlp_test_rmse": mlp_rmse,
    "mlp_test_r2": mlp_r2,
    "boost_test_rmse": boost_rmse,
    "boost_test_r2": boost_r2,
})
run.finish()

**Вывод:**

На тесте MLP получил RMSE около 51 343 и R² около 0.800. HistGradientBoosting показал RMSE около 46 132 и R² около 0.838, то есть на этих данных бустинг оказался точнее. MLP всё равно обучился и объясняет значительную часть разброса цены, но оснований заменять им бустинг нет.

### Задание 6.4. Итоговое резюме

В 5–7 предложениях: превзошла ли нейросеть классические модели на ваших данных; как размер сети и число эпох влияли на результат; что вы возьмёте в проект на зачёт.

**Резюме:**

В практикуме решалась задача регрессии: нужно было предсказать медианную стоимость жилья. Для MLP использовал числовые признаки, one-hot кодирование `ocean_proximity` и стандартизацию числовых данных. Итоговая сеть имела слои 128, 64 и 32 нейрона, а лучшая эпоха по validation — 98. На тесте MLP получил RMSE около 51.3 тыс. и R² около 0.800. Градиентный бустинг оказался лучше: RMSE около 46.1 тыс. и R² около 0.838. Поэтому для проекта оставил бы бустинг основной моделью, а MLP — как дополнительный эксперимент и сравнение.

---

## P.S. Зачем нейросети на табличных данных (и когда нет)

На небольших табличных данных **градиентный бустинг** из занятия 4 обычно не хуже, а часто лучше простого MLP — и работает без подбора архитектуры. Нейросети выигрывают, когда:

- данных много и есть сложные нелинейные зависимости;
- есть категориальные признаки с большой кардинальностью (эмбеддинги);
- задача про **изображения, текст или последовательности** — там работают CNN/LSTM, и это следующий шаг.

Именно поэтому на табличных задачах нейросеть разумно держать как «ещё один кандидат», который сравнивают с бустингом на одной валидации, а не как выбор по умолчанию.

### Справочные материалы

- PyTorch, основы (tutorial): https://pytorch.org/tutorials/beginner/basics/intro.html
- `nn.Module` и слои: https://pytorch.org/docs/stable/nn.html
- Функции потерь: https://pytorch.org/docs/stable/nn.html#loss-functions
- Оптимизаторы (Adam и др.): https://pytorch.org/docs/stable/optim.html